In [1]:
import pandas as pd 
import numpy as np


dev = pd.read_csv("../data/processed/development_processed_v1.csv")
evaluation = pd.read_csv("../data/processed/evaluation_processed_v1.csv")

In [2]:
label_dist = dev['label'].value_counts(normalize= True).sort_index()
label_dist

label
0    0.294286
1    0.132355
2    0.139518
3    0.124717
4    0.107179
5    0.163169
6    0.038776
Name: proportion, dtype: float64

Create new features to chek token e text leng (maybe some political articles, sport one have different distributions)

In [3]:
dev["text_len"] = dev["text"].str.len()
dev["article_len"] = dev["article"].str.len()
dev["title_len"] = dev["title"].fillna("").str.len()
dev["n_tokens"] = dev["text"].str.split().str.len()

In [4]:
evaluation["text_len"] = evaluation["text"].str.len()
evaluation["article_len"] = evaluation["article"].str.len()
evaluation["title_len"] = evaluation["title"].fillna("").str.len()
evaluation["n_tokens"] = evaluation["text"].str.split().str.len()

#check the ratio

In [5]:
dev["title_ratio"] = dev["title_len"] / (dev["text_len"] + 1)


In [6]:
evaluation["title_ratio"] = evaluation["title_len"] / (evaluation["text_len"] + 1)

In [7]:
dev.columns

Index(['Id', 'source', 'title', 'article', 'page_rank', 'timestamp', 'label',
       'year', 'month', 'has_timestamp', 'text', 'text_len', 'article_len',
       'title_len', 'n_tokens', 'title_ratio'],
      dtype='object')

#lexical diversity 

In [8]:
dev["lexical_diversity"] = dev["text"].astype(str).apply(
	lambda x: len(set(x.split())) / (len(x.split()) + 1)
)

In [9]:
evaluation["lexical_diversity"] = evaluation["text"].astype(str).apply(
	lambda x: len(set(x.split())) / (len(x.split()) + 1)
)

In [10]:
dev["text"].map(type).value_counts()

text
<class 'str'>      79996
<class 'float'>        1
Name: count, dtype: int64

In [11]:
length_cols = [
	"text_len",
	"article_len",
	"title_len",
	"n_tokens",
	"title_ratio"
]
dev[length_cols].describe(percentiles=[.1,.25,.5,.75,.9])

,text_len,article_len,title_len,n_tokens,title_ratio
count,79996.000000,79996.000000,79997.000000,79996.000000,79996.000000
mean,227.859930,273.155845,43.765141,37.331354,0.228314
std,214.211038,352.798041,14.133442,35.125431,0.138110
min,2.000000,1.000000,0.000000,1.000000,0.000000
10%,130.000000,100.000000,29.000000,21.000000,0.129139
25%,170.000000,140.000000,33.000000,28.000000,0.163121
50%,223.000000,192.000000,43.000000,36.000000,0.203209
75%,264.000000,239.000000,51.000000,43.000000,0.245614
90%,293.000000,725.000000,61.000000,49.000000,0.309405
max,10958.000000,15790.000000,255.000000,1910.000000,1.481481


In [12]:
dev["lexical_diversity"].describe(percentiles=[.1,.25,.5,.75,.9])

count    79997.000000
mean         0.838453
std          0.064184
min          0.374150
10%          0.757576
25%          0.800000
50%          0.843137
75%          0.883721
90%          0.916667
max          0.984848
Name: lexical_diversity, dtype: float64

In [13]:
num_features = [
	"text_len",
	"article_len",
	"n_tokens",
	"title_len",
	"title_ratio",
	"lexical_diversity",
	"year",
	"month",
	"has_timestamp"
]
corr = dev[num_features].corr()
corr.round(2)

,text_len,article_len,n_tokens,title_len,title_ratio,lexical_diversity,year,month,has_timestamp
text_len,1.00,0.76,1.00,0.13,-0.29,-0.32,0.01,-0.01,0.03
article_len,0.76,1.00,0.76,0.17,-0.17,-0.24,0.09,-0.09,0.19
n_tokens,1.00,0.76,1.00,0.12,-0.28,-0.33,0.01,-0.01,0.04
title_len,0.13,0.17,0.12,1.00,0.20,-0.14,0.03,-0.04,0.08
title_ratio,-0.29,-0.17,-0.28,0.20,1.00,0.13,0.03,-0.04,0.04
lexical_diversity,-0.32,-0.24,-0.33,-0.14,0.13,1.00,0.01,-0.01,-0.01
year,0.01,0.09,0.01,0.03,0.03,0.01,1.00,-0.51,-0.31
month,-0.01,-0.09,-0.01,-0.04,-0.04,-0.01,-0.51,1.00,-0.14
has_timestamp,0.03,0.19,0.04,0.08,0.04,-0.01,-0.31,-0.14,1.00


In [14]:
from sklearn.feature_selection import mutual_info_classif

mi_features = [
	"text_len",
	"n_tokens",
	"title_len",
	"title_ratio",
	"lexical_diversity",
	"year",
	"month",
	"has_timestamp"
]

X_mi = dev[mi_features].fillna(0)
y = dev["label"]

mi_scores = mutual_info_classif(
	X_mi,
	y,
	random_state=42,
	discrete_features=False
)

mi_df = (
	pd.DataFrame({
		"feature": mi_features,
		"mutual_information": mi_scores
	})
	.sort_values("mutual_information", ascending=False)
)

mi_df

,feature,mutual_information
3,title_ratio,0.120621
0,text_len,0.087559
1,n_tokens,0.071136
2,title_len,0.048795
4,lexical_diversity,0.030871
6,month,0.018521
5,year,0.014731
7,has_timestamp,0.010613


From feature information, noone feature dominate the target. 
The text is tme most strong font of information. 
Some stilistich feature add complementar signal. 

Correlation analysis reveals strong redundancy among alternative measures of document length, while mutual information analysis indicates that only a small subset of numeric features provides measurable association with the target labels. In particular, token count and title-to-text ratio capture complementary stylistic information, whereas lexical diversity and raw length measures contribute marginally. Based on these findings, a compact and interpretable feature set is selected for modeling, combining textual representations with a limited number of auxiliary metadata features.

In [15]:
final_features = [
	"text",
	"source",
	"n_tokens",
	"title_ratio",
	"year",
	"month",
	"has_timestamp"
]

Check textual clustering, but not adding other features at the v1 dataset 

In [16]:
dev_eda = dev[dev["text"].notna()].copy()

print("EDA shape:", dev_eda.shape)


EDA shape: (79996, 17)


In [17]:
# STEP 0 — Safety checks & EDA view (no NaN in text)

required_cols = [
	"text", "source", "n_tokens", "title_ratio",
	"year", "month", "has_timestamp", "label"
]

missing_cols = [c for c in required_cols if c not in dev.columns]
print("Missing columns:", missing_cols)

dev_eda = dev[dev["text"].notna()].copy()
print("Original dev shape:", dev.shape)
print("EDA dev_eda shape:", dev_eda.shape)

# Quick check types / NaNs
print("NaN in text (dev_eda):", dev_eda["text"].isna().sum())
dev_eda[required_cols].head(3)


Missing columns: []
Original dev shape: (79997, 17)
EDA dev_eda shape: (79996, 17)
NaN in text (dev_eda): 0


,text,source,n_tokens,title_ratio,year,month,has_timestamp,label
0,opec boosts nigeria's oil revenue by .82m bpd ...,AllAfrica.com,43.0,0.187739,2004.0,9.0,1,5
1,yearender: mideast peace roadmap reaches dead-...,Xinhua,35.0,0.259091,2004.0,12.0,1,0
2,battleground dispatches for oct. 5 \ (cqpoliti...,Yahoo,37.0,0.248945,2006.0,10.0,1,0


In [18]:
# STEP 1 — TF-IDF on text (EDA only)

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_eda = TfidfVectorizer(
	max_features=5000,
	min_df=10,
	stop_words="english"
)

X_text = tfidf_eda.fit_transform(dev_eda["text"])

print("TF-IDF shape:", X_text.shape)
print("TF-IDF nnz:", X_text.nnz)


TF-IDF shape: (79996, 5000)
TF-IDF nnz: 1270845


In [19]:
# STEP 2 — Numeric features (scale)

from sklearn.preprocessing import StandardScaler

num_features = ["n_tokens", "title_ratio", "year", "month", "has_timestamp"]

X_num = dev_eda[num_features].fillna(0)

scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num)

print("Numeric matrix shape:", X_num_scaled.shape)

Numeric matrix shape: (79996, 5)


In [20]:
# STEP 3 — Source reduced + OneHot (EDA-safe)

from sklearn.preprocessing import OneHotEncoder

top_sources = dev_eda["source"].value_counts().head(20).index
dev_eda["source_reduced"] = dev_eda["source"].where(
	dev_eda["source"].isin(top_sources),
	other="OTHER"
)

encoder = OneHotEncoder(
	handle_unknown="ignore",
	sparse_output=True
)
X_source = encoder.fit_transform(dev_eda[["source_reduced"]])

print("OneHot source shape:", X_source.shape)
print("Top sources:", list(top_sources[:5]), "...")

OneHot source shape: (79996, 21)
Top sources: ['Yahoo', 'Reuters', 'BBC', 'New', 'Washington'] ...


In [21]:
# STEP 4 — Build final EDA matrix

from scipy.sparse import hstack

X_eda = hstack([X_text, X_source, X_num_scaled])
print("Final EDA matrix shape:", X_eda.shape)


Final EDA matrix shape: (79996, 5026)


In [22]:
# STEP 5 — KMeans clustering (EDA only)

from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=7, random_state=42, n_init=10)
dev_eda["cluster"] = kmeans.fit_predict(X_eda)

dev_eda["cluster"].value_counts().sort_index()

cluster
0    15852
1    26740
2     2757
3      258
4     9424
5    14117
6    10848
Name: count, dtype: int64

In [23]:
# STEP 6 — Cluster vs label distribution

import pandas as pd

pd.crosstab(dev_eda["cluster"], dev_eda["label"], normalize="index")

label,0,1,2,3,4,5,6
cluster,,,,,,,
0,0.334658,0.136387,0.176129,0.099420,0.077908,0.139793,0.035705
1,0.303553,0.116604,0.089192,0.150898,0.139379,0.156395,0.043979
2,0.030468,0.343489,0.106275,0.019587,0.003264,0.496554,0.000363
3,0.003876,0.000000,0.988372,0.007752,0.000000,0.000000,0.000000
4,0.321413,0.126273,0.200127,0.092848,0.074278,0.152483,0.032576
5,0.309981,0.129631,0.187646,0.110718,0.086633,0.136785,0.038606
6,0.242349,0.123617,0.083057,0.172566,0.154867,0.176807,0.046737


In [24]:
# STEP 7 — Cluster profiling (final numeric features)

dev_eda.groupby("cluster")[["n_tokens","title_ratio","year","month","has_timestamp"]].median()


,n_tokens,title_ratio,year,month,has_timestamp
cluster,,,,,
0,37.0,0.212121,2006.0,10.0,1.0
1,37.0,0.191304,2007.0,8.0,0.0
2,8.0,0.891892,2007.0,8.0,1.0
3,466.5,0.014776,2007.0,7.0,1.0
4,38.0,0.214651,2007.0,6.0,1.0
5,36.0,0.211429,2008.0,2.0,1.0
6,37.0,0.185039,2004.0,10.0,1.0


In [25]:
# STEP 8 — Source distribution by cluster (reduced)

pd.crosstab(dev_eda["cluster"], dev_eda["source_reduced"])


source_reduced,ABC,BBC,Boston,CNET,CNN,Forbes,Guardian,InfoWorld,International,Motley,...,OTHER,RedNova,Register,Reuters,San,Time,Topix.Net,Washington,Wired,Yahoo
cluster,,,,,,,,,,,,,,,,,,,,,
0,0,2136,447,384,214,43,222,80,0,317,...,794,629,264,3182,0,204,99,736,175,4184
1,380,2426,522,240,330,290,238,104,169,178,...,10168,429,97,3072,344,79,763,954,74,4367
2,0,4,9,24,803,133,5,0,484,34,...,688,110,70,13,0,94,0,141,8,103
3,0,0,0,0,0,0,0,253,0,0,...,1,0,0,0,0,0,1,0,0,1
4,0,1198,236,418,98,44,219,34,0,191,...,490,534,151,1898,0,91,20,449,51,2676
5,0,1785,338,252,187,53,206,56,2,144,...,718,718,220,2693,0,195,153,680,197,3582
6,210,422,255,51,170,157,121,27,101,67,...,5425,281,32,1057,196,31,295,422,19,937


The exploratory clustering analysis revealed that certain article categories are strongly associated with specific textual structures, such as long-form articles or headline-dominated news. However, the majority of clusters exhibit overlapping label distributions, indicating that the classification task cannot be solved through simple heuristics or shallow features alone.

This justifies the use of supervised NLP models, particularly transformer-based architectures, while lightweight metadata features are retained as auxiliary signals rather than primary discriminators.

Why: Cluster 3: 
label 2 ≈ 99%
n_tokens ≈ 466
title_ratio ≈ 0.015

Very long articles, title irrelevant, one class dominate 

Cluster 2: 
label 5 ≈ 49%
label 1 ≈ 34%
n_tokens ≈ 8
title_ratio ≈ 0.89
Very shor article probably new flash/headlines /short updates 
Cluster: 0,4,5:

label 0 ≈ 30–33%
label 2 ≈ 17–20%
label 5 ≈ 13–15%
Standard news, avg contex, task ambigus. 


Cluster 6: 
label 3,4,5 equally distributed
no dominace class, probabily thematical mix or temporal mix. Dataset no easy separet. 



In [26]:
##TOP Term per cluster and 2D graph to do 


In [27]:
columns_to_keep = [
	"Id",
	"text",              
	"source",
	"title",
	"n_tokens",
	"title_ratio",
	"year",
	"month",
	"has_timestamp",
	"label"              
]

In [29]:
dev_processed = dev[columns_to_keep].copy()


dev_processed = dev[columns_to_keep].copy()
dev_processed.to_csv(
	"../data/processed/development_v1.csv",
	index=False
)

eval_cols = [c for c in columns_to_keep if c != "label"]
eval_processed = evaluation[eval_cols].copy()
eval_processed.to_csv(
	"../data/processed/evaluation_v1.csv",
	index=False
)